# Using Open Source LLMs with LangChain

Here we will see briefly how you can use popular commercial LLM APIs with LangChain including

- OpenAI GPT (Paid)
- Google Gemini (Paid and Free)

## Install Dependencies

In [ ]:
# Updated LangChain ecosystem packages:
# - langchain==0.4.0 (from 0.3.4)
# - langchain-huggingface==0.1.2 (from 0.1.0)
# - langchain-groq==0.2.1 (from 0.2.0)
# - transformers==4.65.1 (from 4.46.3)

In [1]:
!pip install langchain==0.3.26                                                #0.4.0
!pip install langchain-huggingface==0.3.1                                        # 0.1.2
!pip install langchain-groq==0.3.6
!pip install transformers==4.53.3                                                  # 4.65.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 41.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.7
    Uninstalling langchain-core-1.2.7:
      Successfully uninstalled langchain-core-1.2.7
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.7
    Uninstalling langchain-1.2.7:
      Successfully uninstalled langchain-1.2.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.7 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# Enter API Keys

Here you need to get API keys from the following websites based on your LLM preference:

- Hugging Face Access Token: Go [here](https://huggingface.co/settings/tokens) and create a key with write permissions. You need to setup an account which is totally free of cost.

- Groq API Key: Go [here](https://console.groq.com/keys) and create an API key. You need to setup an account which is totally free of cost. Also while Groq has a generous free tier, there are also paid plans if you are interested.



## Load Hugging Face Access Token Credentials


In [2]:
from getpass import getpass

hf_key = getpass("Enter your Hugging Face Access Token: ")

Enter your Hugging Face Access Token: ··········


## Configure Key in Environment


In [3]:
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_key
os.environ["HF_TOKEN"] = hf_key

## Use LLMs locally with LangChain and Hugging Face

In [4]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

model_id = "meta-llama/Llama-3.2-1B-Instruct"

llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=1000,
        do_sample=False,
        temperature=0,
        return_full_text=False,
    ),
    device=0
)
llm.pipeline.tokenizer.pad_token = llm.pipeline.tokenizer.eos_token
chat_llama = ChatHuggingFace(llm=llm)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [5]:
# Import path simplification in LangChain 0.4.0
# Instead of: from langchain_core.prompts import ChatPromptTemplate
# Now use the simplified path:
from langchain.prompts import ChatPromptTemplate

PROMPT = "Explain {topic} in 2 bullets"
prompt = ChatPromptTemplate.from_template(PROMPT)

chain = (
         prompt
           |
         chat_llama
)

response = chain.invoke({"topic": "AI"})
print(response.content)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Here are 2 bullets explaining AI:

• **Artificial Intelligence (AI)** is a computer system that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making, without being explicitly programmed.

• **AI systems use algorithms and data to analyze and process information, allowing them to make predictions, classify objects, and generate insights that can be used in various applications, including image recognition, natural language processing, and predictive analytics.**


## Use LLMs with LangChain and Hugging Face Inference APIs

In [6]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

model_id = "meta-llama/Llama-3.2-1B-Instruct"

llm_api = HuggingFaceEndpoint(
    repo_id=model_id,
    task="text-generation",
    max_new_tokens=1000,
    do_sample=False,
    temperature=0,
)

chat_llama = ChatHuggingFace(llm=llm_api)

In [7]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT = "Explain {topic} in 2 bullets"
prompt = ChatPromptTemplate.from_template(PROMPT)

chain = (
         prompt
           |
         chat_llama
)

response = chain.invoke({"topic": "AI"})
print(response.content)

Here are two bullets explaining AI:

• **Artificial Intelligence (AI)** refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as learning, problem-solving, decision-making, and perception. AI systems can analyze data, recognize patterns, and make predictions or decisions without being explicitly programmed.

• **Key Areas of Application**: AI has numerous applications across various fields, including but not limited to: virtual assistants like Siri and Alexa, image recognition and facial recognition, natural language processing for chatbots and language translation, predictive analytics for business decision-making, and autonomous vehicles that can navigate and control complex systems.


## Load Groq API Credentials


In [8]:
from getpass import getpass

groq_key = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


## Configure Key in Environment


In [9]:
import os

os.environ["GROQ_API_KEY"] = groq_key

## Use LLMs with LangChain and Groq API

In [10]:
from langchain_groq import ChatGroq

chat_llama = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=1000,
)

In [11]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT = "Explain {topic} in 2 bullets"
prompt = ChatPromptTemplate.from_template(PROMPT)

chain = (
         prompt
           |
         chat_llama
)

response = chain.invoke({"topic": "AI"})
print(response.content)

Here are 2 bullets explaining AI:

* **Definition and Purpose**: Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and perception. The primary purpose of AI is to create machines that can think and act like humans, automating tasks and improving efficiency in various industries.
* **Types and Applications**: AI can be categorized into different types, including narrow or weak AI (e.g., chatbots, virtual assistants), general or strong AI (e.g., human-like intelligence), and superintelligence (e.g., exceeding human intelligence). AI has numerous applications across industries, such as healthcare (diagnosis, treatment), finance (predictive analytics), transportation (self-driving cars), and education (personalized learning), among others.
